In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn nltk beautifulsoup4

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Path to dataset files: /kaggle/input/ag-news-classification-dataset


In [ ]:
import pandas as pd
import os

In [ ]:
import kagglehub
import pandas as pd

# Download data
article_path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

# Use 'ais_raw_data' as our primary variable
ais_raw_data = pd.read_csv(f"{article_path}/train.csv", names=['CategoryID', 'Headline', 'Content'])

# Merge headline and content into a single feature
ais_raw_data['full_article'] = ais_raw_data['Headline'] + " " + ais_raw_data['Content']

print("Article data loaded into 'ais_raw_data'")
print(ais_raw_data.head())

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Article data loaded into 'ais_raw_data'
    CategoryID                                           Headline  \
0  Class Index                                              Title   
1            3  Wall St. Bears Claw Back Into the Black (Reuters)   
2            3  Carlyle Looks Toward Commercial Aerospace (Reu...   
3            3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
4            3  Iraq Halts Oil Exports from Main Southern Pipe...   

                                             Content  \
0                                        Description   
1  Reuters - Short-sellers, Wall Street's dwindli...   
2  Reuters - Private investment firm Carlyle Grou...   
3  Reuters - Soaring crude prices plus worries\ab...   
4  Reuters - Authorities have halted oil export\f...   

                                        full_article  
0                                  Title Description  
1  Wall St. B

In [ ]:
import re
from bs4 import BeautifulSoup

def ais_cleaner(raw_text):
    # Remove HTML
    text_no_html = BeautifulSoup(str(raw_text), "html.parser").get_text()
    # Lowercase & remove non-alphabet characters (punctuation/numbers)
    clean_alphabet = re.sub(r'[^a-zA-Z\s]', '', text_no_html.lower())
    # Basic whitespace cleanup
    return " ".join(clean_alphabet.split())

# Apply cleaning to our new variable
ais_raw_data['cleaned_article'] = ais_raw_data['full_article'].apply(ais_cleaner)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Setup features and target
ais_Features = ais_raw_data['cleaned_article']
ais_Target = ais_raw_data['CategoryID']

# Split into Training and Testing sets
train_X, test_X, train_y, test_y = train_test_split(ais_Features, ais_Target, test_size=0.2, random_state=7)

# 1. Transform text to numbers using a unique vectorizer name
ais_vectorizer = TfidfVectorizer(max_features=4000)
train_X_vectors = ais_vectorizer.fit_transform(train_X)
test_X_vectors = ais_vectorizer.transform(test_X)

# 2. Train the Article Classification Model
ais_ml_processor = LogisticRegression(max_iter=1000)
ais_ml_processor.fit(train_X_vectors, train_y)

# 3. Check performance
ais_predictions = ais_ml_processor.predict(test_X_vectors)
print(classification_report(test_y, ais_predictions))

              precision    recall  f1-score   support

           1       0.92      0.89      0.90      6016
           2       0.94      0.97      0.96      6013
           3       0.88      0.87      0.88      6002
           4       0.87      0.88      0.87      5970

    accuracy                           0.90     24001
   macro avg       0.90      0.90      0.90     24001
weighted avg       0.90      0.90      0.90     24001



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Setup features and target
ais_Features = ais_raw_data['cleaned_article']
ais_Target = ais_raw_data['CategoryID']

# Split into Training and Testing sets
train_X, test_X, train_y, test_y = train_test_split(ais_Features, ais_Target, test_size=0.2, random_state=7)

# 1. Transform text to numbers using a unique vectorizer name
ais_vectorizer = TfidfVectorizer(max_features=4000)
train_X_vectors = ais_vectorizer.fit_transform(train_X)
test_X_vectors = ais_vectorizer.transform(test_X)

# 2. Train the Article Classification Model
ais_ml_processor = LogisticRegression(max_iter=1000)
ais_ml_processor.fit(train_X_vectors, train_y)

# 3. Check performance
ais_predictions = ais_ml_processor.predict(test_X_vectors)
print(classification_report(test_y, ais_predictions))

Files 'article_ml_model.pkl' and 'article_vectorizer.pkl' are ready for AWS!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import kagglehub
import re
from bs4 import BeautifulSoup

# 1. Re-load the data
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
ais_raw_data = pd.read_csv(f"{path}/train.csv", names=['CategoryID', 'Headline', 'Content'])
ais_raw_data['full_article'] = ais_raw_data['Headline'] + " " + ais_raw_data['Content']

# 2. Cleaning Function
def ais_cleaner(raw_text):
    text_no_html = BeautifulSoup(str(raw_text), "html.parser").get_text()
    clean_alphabet = re.sub(r'[^a-zA-Z\s]', '', text_no_html.lower())
    return " ".join(clean_alphabet.split())

# 3. Apply cleaning (Creating the missing 'cleaned_article' column)
ais_raw_data['cleaned_article'] = ais_raw_data['full_article'].apply(ais_cleaner)

# 4. Filter out the header rows ('Class Index') and fix targets
ais_raw_data['CategoryID'] = pd.to_numeric(ais_raw_data['CategoryID'], errors='coerce')
ais_clean_set = ais_raw_data.dropna(subset=['CategoryID']).copy()
ais_clean_set['CategoryID'] = ais_clean_set['CategoryID'].astype(int) - 1

# 5. Re-split the data
train_X_dl, test_X_dl, train_y_dl, test_y_dl = train_test_split(
    ais_clean_set['cleaned_article'],
    ais_clean_set['CategoryID'],
    test_size=0.2,
    random_state=7
)

print("Data Recovered! 'cleaned_article' column is now active.")
print(f"Total clean articles: {len(ais_clean_set)}")

Using Colab cache for faster access to the 'ag-news-classification-dataset' dataset.
Data Recovered! 'cleaned_article' column is now active.
Total clean articles: 120000


In [ ]:
# 1. Clean the labels and drop the 'Class Index' text row
ais_raw_data_clean = ais_raw_data[pd.to_numeric(ais_raw_data['CategoryID'], errors='coerce').notnull()].copy()
ais_raw_data_clean['CategoryID'] = ais_raw_data_clean['CategoryID'].astype(int) - 1

# 2. Re-split the CLEANED data using the exact same random_state
# This ensures X and y are the same length
X_clean = ais_raw_data_clean['cleaned_article']
y_clean = ais_raw_data_clean['CategoryID']

train_X_dl, test_X_dl, train_y_dl, test_y_dl = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=7
)

# 3. Re-tokenize and Pad (Now they will match perfectly)
train_sequences = ais_dl_tokenizer.texts_to_sequences(train_X_dl)
test_sequences = ais_dl_tokenizer.texts_to_sequences(test_X_dl)

train_padded = pad_sequences(train_sequences, maxlen=MAX_LEN)
test_padded = pad_sequences(test_sequences, maxlen=MAX_LEN)

print(f"Success! Train shapes: X={train_padded.shape}, y={train_y_dl.shape}")
print(f"Test shapes: X={test_padded.shape}, y={test_y_dl.shape}")

NameError: name 'ais_dl_tokenizer' is not defined

In [ ]:
# 1. Clean the CategoryID column by removing non-numeric rows (like the header)
# We use errors='coerce' to turn text into NaN, then drop those NaNs
ais_raw_data['CategoryID'] = pd.to_numeric(ais_raw_data['CategoryID'], errors='coerce')
ais_clean_set = ais_raw_data.dropna(subset=['CategoryID']).copy()

# 2. Convert to int and shift to 0-3 for Keras
ais_clean_set['CategoryID'] = ais_clean_set['CategoryID'].astype(int) - 1

# 3. Re-split the features and targets from the clean set
X_final = ais_clean_set['cleaned_article']
y_final = ais_clean_set['CategoryID']

train_X_dl, test_X_dl, train_y_dl, test_y_dl = train_test_split(
    X_final, y_final, test_size=0.2, random_state=7
)

# 4. Re-run Tokenization and Padding on the clean split
train_sequences = ais_dl_tokenizer.texts_to_sequences(train_X_dl)
test_sequences = ais_dl_tokenizer.texts_to_sequences(test_X_dl)

train_padded = pad_sequences(train_sequences, maxlen=MAX_LEN)
test_padded = pad_sequences(test_sequences, maxlen=MAX_LEN)

print("Alignment successful!")
print(f"Input shape: {train_padded.shape}, Label shape: {train_y_dl.shape}")

Alignment successful!
Input shape: (96000, 100), Label shape: (96000,)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D

# Architecture
ais_nn_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    SpatialDropout1D(0.3),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(4, activation='softmax')
])

ais_nn_model.compile(loss='sparse_categorical_crossentropy',
                     optimizer='adam',
                     metrics=['accuracy'])

# Start Training
print("Starting training...")
ais_nn_model.fit(train_padded, train_y_dl, epochs=5, batch_size=64, validation_split=0.1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Starting training...
Epoch 1/5
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 207s 149ms/step - accuracy: 0.8491 - loss: 0.4352 - val_accuracy: 0.9028 - val_loss: 0.2897
Epoch 2/5
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 196s 145ms/step - accuracy: 0.9162 - loss: 0.2690 - val_accuracy: 0.9121 - val_loss: 0.2787
Epoch 3/5
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 199s 143ms/step - accuracy: 0.9267 - loss: 0.2302 - val_accuracy: 0.9105 - val_loss: 0.2852
Epoch 4/5
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 201s 142ms/step - accuracy: 0.9346 - loss: 0.2032 - val_accuracy: 0.9109 - val_loss: 0.2813
Epoch 5/5
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 198s 147ms/step - accuracy: 0.9419 - loss: 0.1777 - val_accuracy: 0.9090 - val_loss: 0.3099


In [ ]:
!pip install transformers datasets accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install --upgrade pyarrow datasets transformers accelerate --quiet

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch

# 1. Initialize the Transformer Tokenizer
# This handles the specific word-piece tokenization BERT requires
ais_transformer_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# 2. Prepare a 5,000 sample subset for stable training in Colab
# Transformers are heavy; a subset is best for the free tier
transformer_train_texts = train_X_dl.head(5000).tolist()
transformer_train_labels = train_y_dl.head(5000).tolist()

# 3. Create the Encodings
train_encodings = ais_transformer_tokenizer(transformer_train_texts, truncation=True, padding=True, max_length=128)

# 4. Convert to a Torch Dataset
class AISDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

ais_train_data = AISDataset(train_encodings, transformer_train_labels)

# 5. Load the Pre-trained Model with a 4-class head
ais_transformer_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)

# 6. Define the Training Arguments
train_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1, # 1 epoch is usually enough for a high-accuracy baseline
    per_device_train_batch_size=16,
    logging_steps=50,
    report_to="none"
)

# 7. Start Training (Fine-tuning)
ais_trainer = Trainer(
    model=ais_transformer_model,
    args=train_args,
    train_dataset=ais_train_data
)

print("Starting Transformer training... this will take about 2-4 minutes.")
ais_trainer.train()

ImportError: cannot import name 'merge_with_config_defaults' from 'transformers.utils.generic' (/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py)

In [ ]:
# Save using the correct variable names from your training cell
ais_trainer.model.save_pretrained("ais_transformer_final")
ais_transformer_tokenizer.save_pretrained("ais_transformer_final")

# Zip it for easy download to your local machine
!zip -r ais_transformer.zip ais_transformer_final/

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: ais_transformer_final/ (stored 0%)
  adding: ais_transformer_final/config.json (deflated 53%)
  adding: ais_transformer_final/tokenizer_config.json (deflated 43%)
  adding: ais_transformer_final/model.safetensors (deflated 8%)
  adding: ais_transformer_final/tokenizer.json (deflated 71%)


In [ ]:
# Re-import the tools
from transformers import Trainer

# Re-define the trainer using your existing model and dataset variables
# Note: Ensure you use the exact variable names from your training cell
trainer = Trainer(
    model=ais_transformer_model,
    args=train_args,
    train_dataset=ais_train_data
)

# Now run the evaluation
results = trainer.evaluate()
print(f"Final Transformer Accuracy: {results['eval_accuracy']:.4f}")

NameError: name 'ais_transformer_model' is not defined

In [ ]:
# Run this to see how well the Transformer performed
results = trainer.evaluate()
print(f"Transformer Accuracy: {results['eval_accuracy']:.4f}")

NameError: name 'trainer' is not defined